# Context Engineering 完整演示

本 notebook 用纯 Python 演示上下文工程的核心流程：上下文块、窗口管理、记忆、RAG、注入策略、压缩与评估。

建议把它当成“最小工程骨架”来看，而不是完整生产系统。阅读时重点关注三个问题：

1. 模型到底需要哪些信息，而不是把所有资料都塞进去。
2. 这些信息应该如何分层、排序、裁剪和注入。
3. 如何判断上下文是否真的帮到了回答质量，而不是只让 prompt 更长。

配合课程 PPT 和讲义，这个 notebook 最适合承担两件事：

- 把抽象概念变成可运行的 Python 对象和函数。
- 用小规模示例提前暴露真实系统中的常见问题，例如窗口超限、检索误召回、压缩丢关键信息、引用冲突等。


In [19]:
from dataclasses import dataclass, field
from typing import Dict, List, Tuple
from collections import Counter
import math
import re

# 下面的资料片段来自 docs/上下文工程.pptx，用作 notebook 的真实课程语料。
COURSE_SOURCES = {
    "slide3_context_definition": "什么是上下文工程？上下文工程是设计、构建和管理AI系统上下文的工程实践。管理对话历史和状态；控制上下文窗口的有效利用；优化token使用成本；确保AI响应的相关性和准确性。",
    "slide5_context_lifecycle": "上下文管理基础：上下文生命周期管理，创建 → 累积 → 压缩 → 清除。滑动窗口策略：保留最近N轮对话，自动丢弃旧内容。重要性加权：根据消息类型和时间重要性动态调整。",
    "slide6_memory_architecture": "记忆系统架构：短时记忆是当前对话窗口；工作记忆是会话期间重要信息；长时记忆是持久化存储知识。核心组件包括记忆存储层、检索索引系统、记忆压缩模块、遗忘机制。",
    "slide7_memory_types": "记忆类型详解：语义记忆是事实性知识、概念定义、通用常识；情景记忆是具体事件、对话经历、用户偏好；程序记忆是操作流程、技能方法、执行策略。",
    "slide8_rag_principle": "RAG 检索增强生成：结合检索系统与语言模型，突破知识截止限制。工作流程：用户查询 → 向量检索 → 上下文增强 → LLM生成。优势：实时知识更新、减少幻觉、提高可溯源性。",
    "slide9_rag_components": "RAG 系统组件：数据源包括文档、数据库、API；分块处理包括文本分割、语义分块；向量化使用Embedding模型；向量数据库包括Milvus、Pinecone等；检索器做相似度搜索、重排序；生成器是LLM推理引擎。",
    "slide10_injection_methods": "上下文注入技术：直接注入是将上下文直接拼接至prompt；压缩注入是摘要压缩后注入，减少token；动态注入是根据查询意图选择性注入；分层注入是先检索后精调，层层过滤。",
    "slide11_prompt_context": "Prompt 工程与上下文：System Prompt 设计定义AI角色、知识范围、响应格式；Few-shot Examples通过示例引导AI理解任务模式；上下文格式化使用结构化输出、JSON格式、清晰分隔。",
    "slide12_token_budget": "上下文窗口管理：Token预算规划策略包括预留空间、动态分配、溢出处理、成本监控。预留空间是为系统指令和检索结果预留固定token；动态分配是根据对话阶段调整用户/AI比例；溢出处理是在超过阈值时触发摘要或截断；成本监控是追踪token消耗，优化利用率。",
    "slide13_best_practices": "最佳实践：保持上下文简洁，避免冗余信息；使用结构化格式提高可读性；定期清理无用对话历史；监控token使用，控制成本；根据场景选择合适的上下文策略；测试不同上下文配置的效果。",
    "slide14_scenarios": "应用场景：智能客服需要多轮对话理解、个性化服务；代码助手需要项目上下文、代码解释生成；文档助手需要长文档摘要、问答系统；数据分析需要上下文理解、数据可视化建议。",
}


## 1. 上下文块与优先级

把输入拆成可管理的 `ContextBlock`，后续才能做裁剪、排序、注入和评估。

如果系统提示、检索资料、历史对话、用户偏好都直接拼成一个大字符串，会马上遇到三个工程问题：

- 无法区分哪段信息最重要。
- 超过窗口时不知道该删哪一段。
- 回答出错时很难定位是“资料给错了”还是“模型理解错了”。

因此更合理的做法是先把上下文对象化。一个上下文块至少要能表达：

- `name`：它属于系统规则、知识片段、历史消息还是用户画像。
- `priority`：窗口不足时谁应该优先保留。
- `source`：内容从哪里来，便于引用和调试。
- `tokens`：它大概会占多少输入预算。

下面这个例子故意保持极简，但已经体现了上下文工程和提示词工程的差异：重点不在“句子怎么写得更漂亮”，而在“先把信息组织清楚”。


In [20]:
@dataclass
class ContextBlock:
    name: str
    content: str
    priority: int = 1
    source: str = "manual"

    @property
    def tokens(self) -> int:
        chinese = sum(1 for c in self.content if "\u4e00" <= c <= "\u9fff")
        other = len(self.content) - chinese
        return int(chinese * 1.5 + other * 0.25)

blocks = [
    ContextBlock("system", "你是上下文工程课程助手，只基于课程PPT资料回答。", priority=5, source="course_rule"),
    ContextBlock("definition", COURSE_SOURCES["slide3_context_definition"], priority=4, source="ppt_slide3"),
    ContextBlock("rag", COURSE_SOURCES["slide8_rag_principle"], priority=4, source="ppt_slide8"),
    ContextBlock("lifecycle", COURSE_SOURCES["slide5_context_lifecycle"], priority=3, source="ppt_slide5"),
    ContextBlock("scenarios", COURSE_SOURCES["slide14_scenarios"], priority=2, source="ppt_slide14"),
]

for block in blocks:
    print(block.name, block.priority, block.tokens, block.source, block.content[:45] + "...")


system 5 31 course_rule 你是上下文工程课程助手，只基于课程PPT资料回答。...
definition 4 109 ppt_slide3 什么是上下文工程？上下文工程是设计、构建和管理AI系统上下文的工程实践。管理对话历史和状态...
rag 4 99 ppt_slide8 RAG 检索增强生成：结合检索系统与语言模型，突破知识截止限制。工作流程：用户查询 → 向...
lifecycle 3 102 ppt_slide5 上下文管理基础：上下文生命周期管理，创建 → 累积 → 压缩 → 清除。滑动窗口策略：保留...
scenarios 2 108 ppt_slide14 应用场景：智能客服需要多轮对话理解、个性化服务；代码助手需要项目上下文、代码解释生成；文档...


## 2. Token 预算与窗口管理

上下文窗口大，不代表可以无上限地塞资料。真正要管理的是预算，而不是长度幻觉。

在一个实际系统里，输入预算通常至少要分给四部分：

- 系统指令和输出格式要求
- 当前用户问题
- 检索或记忆返回的上下文
- 预留给模型生成答案的空间

如果不做预算规划，典型后果有三个：

- 回答在输出阶段被截断
- 检索资料过多，真正的问题反而被淹没
- token 成本和延迟明显上升

下面的函数只演示最简单的“按优先级选入预算”。真实系统还会再加：

- 重要性和时效性的联合排序
- 按 token 而不是按字符的更精确估算
- 超限时先摘要、再截断，而不是直接删除


In [37]:
def select_context(blocks: List[ContextBlock], budget: int) -> List[ContextBlock]:
    selected = []
    used = 0
    for block in sorted(blocks, key=lambda x: x.priority, reverse=True):
        if used + block.tokens <= budget:
            selected.append(block)
            used += block.tokens
    return selected

selected = select_context(blocks, budget=170)
print("选中的上下文：")
for block in selected:
    print("-", block.name, block.tokens)


选中的上下文：
- system 31
- definition 109


## 2.1 上下文生命周期管理与滑动窗口

PPT 里的“创建 → 累积 → 压缩 → 清除”适合用一个课程问答助手来讲，而不是只演示“保留最近 N 条消息”。

在真实应用中，一条上下文通常会经历这样的生命周期：

1. 创建：用户提问、系统规则、检索资料、工具结果进入上下文池。
2. 累积：多轮对话后，上下文池变大，历史、约束、资料混在一起。
3. 压缩：旧对话不再逐字保留，而是变成摘要，减少 token 占用。
4. 清除：过期通知、重复资料、低优先级历史被移出当前窗口。

下面的例子模拟“课程问答机器人”处理一个新问题：它会固定保留系统规则，压缩旧对话，清理过期通知，再按滑动窗口和优先级选择最终注入的上下文。这样更能体现 PPT 里“生命周期管理”和“滑动窗口策略”之间的关系。


In [22]:
from dataclasses import dataclass


@dataclass
class ContextItem:
    kind: str
    content: str
    priority: int
    turn: int
    source: str
    keep: bool = True

    @property
    def tokens(self):
        chinese = sum(1 for c in self.content if "\u4e00" <= c <= "\u9fff")
        other = len(self.content) - chinese
        return int(chinese * 1.5 + other * 0.25)


context_pool = [
    ContextItem('system', '只根据课程PPT资料回答，并说明使用了哪些slide。', 5, 0, 'course_rule'),
    ContextItem('course_source', COURSE_SOURCES['slide5_context_lifecycle'], 5, 1, 'ppt_slide5'),
    ContextItem('course_source', COURSE_SOURCES['slide12_token_budget'], 4, 2, 'ppt_slide12'),
    ContextItem('course_source', COURSE_SOURCES['slide14_scenarios'], 2, 3, 'ppt_slide14', keep=False),
    ContextItem('user', '用户问题：如何讲解上下文生命周期管理和滑动窗口？', 4, 4, 'runtime_user'),
]


def compress_old_course_context(items, current_turn, keep_recent_turns=2):
    old_sources = [item for item in items if item.kind == 'course_source' and current_turn - item.turn > keep_recent_turns and item.keep]
    if not old_sources:
        return []
    summary = '摘要：上下文管理包含创建、累积、压缩、清除；滑动窗口保留最近N轮，重要性加权决定保留顺序。'
    return [ContextItem('summary', summary, priority=4, turn=current_turn, source='summary_from_ppt_slide5')]


def clear_irrelevant(items):
    active = [item for item in items if item.keep]
    removed = [item for item in items if not item.keep]
    return active, removed


def build_sliding_context(items, current_turn, token_budget=220):
    compressed = compress_old_course_context(items, current_turn)
    active_items, removed_items = clear_irrelevant(items)
    recent_or_pinned = [
        item for item in active_items
        if item.kind == 'system' or current_turn - item.turn <= 2
    ]
    candidates = recent_or_pinned + compressed
    selected = []
    used = 0
    for item in sorted(candidates, key=lambda x: (x.priority, x.turn), reverse=True):
        if used + item.tokens <= token_budget:
            selected.append(item)
            used += item.tokens
    return {
        'removed': removed_items,
        'compressed': compressed,
        'candidates': candidates,
        'selected': selected,
        'used_tokens': used,
        'budget': token_budget,
    }


result = build_sliding_context(context_pool, current_turn=4)

print('1. 清除与当前问题弱相关的上下文:')
for item in result['removed']:
    print(f"- [{item.source}] {item.content[:55]}...")

print()
print('2. 压缩较早但仍重要的课程资料:')
for item in result['compressed']:
    print(f"- [{item.source}] {item.content}")

print()
print('3. 进入滑动窗口的候选上下文:')
for item in result['candidates']:
    print(f"- [{item.kind}] source={item.source} p={item.priority} turn={item.turn} tokens={item.tokens}")

print()
print('4. 最终注入上下文:')
for item in result['selected']:
    print(f"- [{item.source}] p={item.priority} tokens={item.tokens} | {item.content[:70]}...")
print(f"token 预算: {result['used_tokens']} / {result['budget']}")


1. 清除与当前问题弱相关的上下文:
- [ppt_slide14] 应用场景：智能客服需要多轮对话理解、个性化服务；代码助手需要项目上下文、代码解释生成；文档助手需要长文档摘要、...

2. 压缩较早但仍重要的课程资料:
- [summary_from_ppt_slide5] 摘要：上下文管理包含创建、累积、压缩、清除；滑动窗口保留最近N轮，重要性加权决定保留顺序。

3. 进入滑动窗口的候选上下文:
- [system] source=course_rule p=5 turn=0 tokens=28
- [course_source] source=ppt_slide12 p=4 turn=2 tokens=155
- [user] source=runtime_user p=4 turn=4 tokens=33
- [summary] source=summary_from_ppt_slide5 p=4 turn=4 tokens=57

4. 最终注入上下文:
- [course_rule] p=5 tokens=28 | 只根据课程PPT资料回答，并说明使用了哪些slide。...
- [runtime_user] p=4 tokens=33 | 用户问题：如何讲解上下文生命周期管理和滑动窗口？...
- [summary_from_ppt_slide5] p=4 tokens=57 | 摘要：上下文管理包含创建、累积、压缩、清除；滑动窗口保留最近N轮，重要性加权决定保留顺序。...
token 预算: 118 / 220


## 3. 短期记忆与长期记忆

只保存聊天记录，不等于拥有记忆系统。记忆的关键不是“存起来”，而是“该存什么、什么时候取回、取回多少”。

在课程里可以先把记忆分成两层：

- 短期记忆：保存最近几轮对话，保证当前会话连贯。
- 长期记忆：保存跨会话仍然有价值的事实、偏好或项目约束。

这两层的典型用途并不一样：

- 短期记忆适合保留“当前正在讨论什么”。
- 长期记忆适合保留“用户是谁、项目有哪些固定约束、历史做过什么关键决策”。

下面的示例把检索逻辑简化成基于标签召回，目的是先说明“记忆应该可检索”，而不是把整段历史原样塞回 prompt。


In [23]:
class MemoryStore:
    def __init__(self):
        self.short_term = []
        self.long_term = []

    def add_turn(self, role, content):
        self.short_term.append({"role": role, "content": content})
        self.short_term = self.short_term[-6:]

    def remember(self, fact, tags=None, source=None):
        self.long_term.append({"fact": fact, "tags": tags or [], "source": source})

    def recall(self, query):
        return [item for item in self.long_term if any(tag in query for tag in item["tags"])]

memory = MemoryStore()
memory.add_turn("user", "PPT 里上下文窗口管理有哪些策略？")
memory.add_turn("assistant", COURSE_SOURCES["slide12_token_budget"])
memory.remember(
    "上下文工程是设计、构建和管理AI系统上下文的工程实践。",
    tags=["核心概念", "上下文工程"],
    source="ppt_slide3",
)

print("短期记忆：保留当前对话的最近轮次")
for turn in memory.short_term:
    print(f"- {turn['role']}: {turn['content'][:80]}...")

print()
print("长期记忆召回：根据标签找回稳定课程概念")
for item in memory.recall("上下文工程核心概念"):
    print(f"- {item['fact']} | source={item['source']} | tags={item['tags']}")


短期记忆：保留当前对话的最近轮次
- user: PPT 里上下文窗口管理有哪些策略？...
- assistant: 上下文窗口管理：Token预算规划策略包括预留空间、动态分配、溢出处理、成本监控。预留空间是为系统指令和检索结果预留固定token；动态分配是根据对话阶段调整用...

长期记忆召回：根据标签找回稳定课程概念
- 上下文工程是设计、构建和管理AI系统上下文的工程实践。 | source=ppt_slide3 | tags=['核心概念', '上下文工程']


## 3.1 记忆类型详解：语义记忆 / 情景记忆 / 程序记忆

PPT 中把记忆进一步分成三类，这一点值得单独举例，否则学生容易把“长期记忆”理解成一个大杂烩。

- 语义记忆：稳定知识，例如“RAG 由检索和生成组成”。
- 情景记忆：发生过的具体事件，例如“昨天学生已经问过引用格式”。
- 程序记忆：操作方法或流程，例如“先检索，再重排，再回答”。

这三类记忆在课堂场景里都很常见，而且检索时的价值也不同：

- 回答概念题时更依赖语义记忆。
- 延续多轮任务时更依赖情景记忆。
- 执行复杂流程时更依赖程序记忆。


In [24]:
memory_examples = {
    'semantic': [
        '语义记忆：事实性知识、概念定义、通用常识。',
    ],
    'episodic': [
        '情景记忆：具体事件、对话经历、用户偏好。',
    ],
    'procedural': [
        '程序记忆：操作流程、技能方法、执行策略。',
    ],
}

print('资料来源: ppt_slide7_memory_types')
for kind, items in memory_examples.items():
    print(f'[{kind}]')
    for item in items:
        print('-', item)


资料来源: ppt_slide7_memory_types
[semantic]
- 语义记忆：事实性知识、概念定义、通用常识。
[episodic]
- 情景记忆：具体事件、对话经历、用户偏好。
[procedural]
- 程序记忆：操作流程、技能方法、执行策略。


## 3.2 记忆系统架构：存储层、索引、压缩与遗忘

PPT 中“记忆系统架构”不只是说有几类记忆，还强调了几个核心组件：

- 存储层：把记忆保存下来。
- 检索索引：让记忆可以被找回来。
- 压缩模块：防止历史无限膨胀。
- 遗忘机制：淘汰陈旧、重复或低价值的记忆。

下面这个简化例子用重要性和访问次数模拟“保留什么、遗忘什么”。


In [25]:
memory_bank = [
    {'fact': '上下文生命周期管理：创建 → 累积 → 压缩 → 清除。', 'importance': 1.0, 'access': 4, 'source': 'ppt_slide5'},
    {'fact': '成本监控：追踪token消耗，优化利用率。', 'importance': 0.9, 'access': 3, 'source': 'ppt_slide12'},
    {'fact': '应用场景：数据分析需要上下文理解、数据可视化建议。', 'importance': 0.4, 'access': 0, 'source': 'ppt_slide14'},
    {'fact': 'Prompt工程包含Few-shot Examples和上下文格式化。', 'importance': 0.7, 'access': 1, 'source': 'ppt_slide11'},
]


def forget_low_value(memories, threshold=0.75):
    kept = []
    dropped = []
    for item in memories:
        score = item['importance'] + 0.05 * item['access']
        row = (score, item['source'], item['fact'])
        if score >= threshold:
            kept.append(row)
        else:
            dropped.append(row)
    return kept, dropped

kept, dropped = forget_low_value(memory_bank)
print('保留的记忆:')
for score, source, fact in kept:
    print(f'- score={score:.2f} | {source} | {fact}')
print()
print('被遗忘的记忆:')
for score, source, fact in dropped:
    print(f'- score={score:.2f} | {source} | {fact}')


保留的记忆:
- score=1.20 | ppt_slide5 | 上下文生命周期管理：创建 → 累积 → 压缩 → 清除。
- score=1.05 | ppt_slide12 | 成本监控：追踪token消耗，优化利用率。
- score=0.75 | ppt_slide11 | Prompt工程包含Few-shot Examples和上下文格式化。

被遗忘的记忆:
- score=0.40 | ppt_slide14 | 应用场景：数据分析需要上下文理解、数据可视化建议。


## 4. 简化 RAG：切分、检索、生成上下文

RAG 的核心不是“查一下资料再回答”，而是把外部知识有控制地引入模型上下文。

一个更完整的 RAG 链路通常包括：

1. 文档预处理
2. 文档分块
3. 向量化或其他索引方式
4. 粗召回
5. 重排序
6. 组装上下文
7. 生成答案与引用
8. 评估检索和回答质量

这个 notebook 里先用一个非常小的文本集合演示“查询和资料如何发生匹配”。重点是让你看到：

- 检索结果本身也是上下文，需要管理。
- 返回 Top-K 不代表一定可用，还要考虑排序、冗余和冲突。
- 同样的问题，分块方式不同，最终送给模型的内容也会不同。


In [26]:
docs = {
    "rag_principle": COURSE_SOURCES["slide8_rag_principle"],
    "rag_components": COURSE_SOURCES["slide9_rag_components"],
    "context_injection": COURSE_SOURCES["slide10_injection_methods"],
    "token_budget": COURSE_SOURCES["slide12_token_budget"],
}

DOMAIN_TERMS = [
    'RAG', '检索增强生成', '工作流程', '核心思想', '优势', '检索系统', '语言模型', '知识截止', '用户查询', '向量检索',
    '上下文增强', 'LLM生成', '实时知识更新', '减少幻觉', '可溯源性', '数据源',
    '文档', '数据库', 'API', '分块处理', '文本分割', '语义分块', 'Embedding',
    '向量数据库', 'Milvus', 'Pinecone', '相似度搜索', '重排序', '上下文注入',
    '直接注入', '压缩注入', '动态注入', '分层注入', 'Token预算', '成本监控'
]


def tokenize(text):
    tokens = set(re.findall(r"[a-zA-Z]+|\d+", text.lower()))
    for term in DOMAIN_TERMS:
        if term.lower() in text.lower():
            tokens.add(term.lower())
    return tokens


def retrieve(query, top_k=2):
    q = tokenize(query)
    scored = []
    for name, text in docs.items():
        score = len(q & tokenize(text)) / max(len(q), 1)
        scored.append((score, name, text))
    return sorted(scored, key=lambda x: x[0], reverse=True)[:top_k]

query = "RAG 的工作流程和系统组件是什么？"
retrieved = retrieve(query)
for score, name, text in retrieved:
    print(f"{name}: {score:.2f} {text[:90]}...")


rag_principle: 1.00 RAG 检索增强生成：结合检索系统与语言模型，突破知识截止限制。工作流程：用户查询 → 向量检索 → 上下文增强 → LLM生成。优势：实时知识更新、减少幻觉、提高可溯源性。...
rag_components: 0.50 RAG 系统组件：数据源包括文档、数据库、API；分块处理包括文本分割、语义分块；向量化使用Embedding模型；向量数据库包括Milvus、Pinecone等；检索器做相似度搜...


## 4.1 分块策略对比：为什么不能只按固定长度切

很多初学者第一次做 RAG，会直接按字符数切块。这能快速跑通流程，但会引入两个问题：

- 语义可能被切断，导致关键信息跨块分裂。
- 命中的是整段制度或整章教材，而不是最相关的小节。

下面用“固定长度分块”和“按标题分块”做一个对比。这个例子很适合课堂讲解，因为它能直观看到：分块方式不同，命中的证据块也不同。


In [27]:
course_doc = """
# 上下文管理基础
上下文生命周期管理：创建 → 累积 → 压缩 → 清除。滑动窗口策略：保留最近N轮对话，自动丢弃旧内容。重要性加权：根据消息类型和时间重要性动态调整。

# RAG 检索增强生成
核心思想：结合检索系统与语言模型，突破知识截止限制。工作流程：用户查询 → 向量检索 → 上下文增强 → LLM生成。优势：实时知识更新、减少幻觉、提高可溯源性。

# 最佳实践
保持上下文简洁，避免冗余信息。使用结构化格式提高可读性。定期清理无用对话历史。监控token使用，控制成本。根据场景选择合适的上下文策略。测试不同上下文配置的效果。
""".strip()


def fixed_chunks(text, size=70, overlap=12):
    chunks = []
    start = 0
    while start < len(text):
        end = start + size
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start = end - overlap
    return chunks


def heading_chunks(text):
    result = []
    current_title = 'untitled'
    current_lines = []
    for line in text.splitlines():
        if line.startswith('#'):
            if current_lines:
                result.append((current_title, ' '.join(current_lines).strip()))
            current_title = line.lstrip('#').strip()
            current_lines = []
        elif line.strip():
            current_lines.append(line.strip())
    if current_lines:
        result.append((current_title, ' '.join(current_lines).strip()))
    return result

chunk_query = 'RAG 的工作流程是什么？'
print('固定长度分块：')
def query_score(text):
    return len(tokenize(chunk_query) & tokenize(text))
for idx, chunk in enumerate(fixed_chunks(course_doc), 1):
    print(f'  chunk-{idx}: score={query_score(chunk)} | {chunk}')

print()
print('按标题分块：')
for title, chunk in heading_chunks(course_doc):
    print(f'  {title}: score={query_score(chunk)} | {chunk}')


固定长度分块：
  chunk-1: score=0 | # 上下文管理基础
上下文生命周期管理：创建 → 累积 → 压缩 → 清除。滑动窗口策略：保留最近N轮对话，自动丢弃旧内容。重要性加权：根据
  chunk-2: score=1 | 旧内容。重要性加权：根据消息类型和时间重要性动态调整。

# RAG 检索增强生成
核心思想：结合检索系统与语言模型，突破知识截止限制。工作
  chunk-3: score=1 | ，突破知识截止限制。工作流程：用户查询 → 向量检索 → 上下文增强 → LLM生成。优势：实时知识更新、减少幻觉、提高可溯源性。

# 最
  chunk-4: score=0 | 提高可溯源性。

# 最佳实践
保持上下文简洁，避免冗余信息。使用结构化格式提高可读性。定期清理无用对话历史。监控token使用，控制成本。
  chunk-5: score=0 | oken使用，控制成本。根据场景选择合适的上下文策略。测试不同上下文配置的效果。

按标题分块：
  上下文管理基础: score=0 | 上下文生命周期管理：创建 → 累积 → 压缩 → 清除。滑动窗口策略：保留最近N轮对话，自动丢弃旧内容。重要性加权：根据消息类型和时间重要性动态调整。
  RAG 检索增强生成: score=1 | 核心思想：结合检索系统与语言模型，突破知识截止限制。工作流程：用户查询 → 向量检索 → 上下文增强 → LLM生成。优势：实时知识更新、减少幻觉、提高可溯源性。
  最佳实践: score=0 | 保持上下文简洁，避免冗余信息。使用结构化格式提高可读性。定期清理无用对话历史。监控token使用，控制成本。根据场景选择合适的上下文策略。测试不同上下文配置的效果。


## 4.2 检索后重排序：Top-K 不是终点

检索系统常见错误不是“完全找不到”，而是“找到了，但最相关的没有排在前面”。这就是为什么生产系统里常常需要 `rerank`。

下面这个示例用一个非常粗糙的规则做重排：

- 先保留原始检索分数
- 再对同时包含关键短语的文档加权

它不能替代真正的 rerank 模型，但足够说明一个重要事实：

检索质量不仅取决于“召回了什么”，也取决于“把什么排在前面”。


In [28]:
def rerank(query, retrieved_docs):
    boosted = []
    for score, name, text in retrieved_docs:
        bonus = 0.0
        if 'RAG' in text and '检索' in text:
            bonus += 0.2
        if '上下文' in text:
            bonus += 0.1
        boosted.append((score + bonus, name, text, score, bonus))
    return sorted(boosted, reverse=True)

raw_results = retrieve('RAG 如何管理检索上下文？', top_k=3)
print('原始排序：')
for score, name, text in raw_results:
    print(f'  {name}: base={score:.2f}')

print('\n重排后：')
for final_score, name, text, base, bonus in rerank('RAG 如何管理检索上下文？', raw_results):
    print(f'  {name}: final={final_score:.2f} | base={base:.2f} | bonus={bonus:.2f}')


原始排序：
  rag_principle: base=1.00
  rag_components: base=1.00
  context_injection: base=0.00

重排后：
  rag_principle: final=1.30 | base=1.00 | bonus=0.30
  rag_components: final=1.20 | base=1.00 | bonus=0.20
  context_injection: final=0.10 | base=0.00 | bonus=0.10


## 5. Prompt 工程与上下文：System Prompt / Few-shot / 格式化

PPT 单独列了这一页，是因为很多学生会把 Prompt Engineering 和 Context Engineering 完全割裂开。实际上两者关系更像：

- Prompt Engineering 决定模型“如何回答”。
- Context Engineering 决定模型“拿什么信息回答”。

真实系统里，这三件事通常一起出现：

- `System Prompt` 规定角色、边界、输出要求。
- `Few-shot` 示例规定任务模式。
- `上下文格式化` 规定资料如何摆放、如何引用、如何分区。

下面用一个小例子把三者放进同一个 prompt。


In [29]:
def build_prompt_with_few_shot(question, retrieved_docs):
    docs_text = "\n".join(f'[{name}] {text}' for _, name, text in retrieved_docs)
    return f"""[System]
你是上下文工程课程助手，只能依据课程PPT资料回答；如果资料不足，要明确说明。
输出格式：先回答结论，再列出依据来源。

[Few-shot]
问题：RAG 的作用是什么？
回答：RAG 结合检索系统与语言模型，通过“用户查询 → 向量检索 → 上下文增强 → LLM生成”增强回答。[依据: rag_principle]

[Context]
{docs_text}

[User]
{question}
"""

few_shot_prompt = build_prompt_with_few_shot('上下文注入有哪些方式？', retrieve('上下文注入有哪些方式？', top_k=2))
print(few_shot_prompt)


[System]
你是上下文工程课程助手，只能依据课程PPT资料回答；如果资料不足，要明确说明。
输出格式：先回答结论，再列出依据来源。

[Few-shot]
问题：RAG 的作用是什么？
回答：RAG 结合检索系统与语言模型，通过“用户查询 → 向量检索 → 上下文增强 → LLM生成”增强回答。[依据: rag_principle]

[Context]
[context_injection] 上下文注入技术：直接注入是将上下文直接拼接至prompt；压缩注入是摘要压缩后注入，减少token；动态注入是根据查询意图选择性注入；分层注入是先检索后精调，层层过滤。
[rag_principle] RAG 检索增强生成：结合检索系统与语言模型，突破知识截止限制。工作流程：用户查询 → 向量检索 → 上下文增强 → LLM生成。优势：实时知识更新、减少幻觉、提高可溯源性。

[User]
上下文注入有哪些方式？



## 6. 上下文注入模板

检索到资料只是第一步，怎么把资料放进 prompt 同样会影响效果。

常见的注入方式至少有四种：

- 直接拼接：实现最快，但模型难以区分来源和层级。
- 标签分隔：让模型知道这是一组“参考资料”。
- 结构化注入：同时提供来源、相关度、字段标签，更适合需要引用的场景。
- 分层注入：高相关资料完整注入，低相关资料只给摘要。

这一节的重点是让学生理解：上下文注入不是简单搬运文本，而是在告诉模型“这些文本之间是什么关系、回答时应该优先依赖谁”。


In [30]:
def build_prompt(question, retrieved_docs, memory_items):
    knowledge = "\n".join(f"[{name}] {text}" for _, name, text in retrieved_docs)
    memories = "\n".join(item["fact"] for item in memory_items) or "无"
    return f"""你是上下文工程课程助手。

长期记忆：
{memories}

检索资料：
{knowledge}

用户问题：{question}

请基于资料回答，缺失信息要说明。"""

prompt = build_prompt(query, retrieved, memory.recall("课程展示"))
print(prompt)


你是上下文工程课程助手。

长期记忆：
无

检索资料：
[rag_principle] RAG 检索增强生成：结合检索系统与语言模型，突破知识截止限制。工作流程：用户查询 → 向量检索 → 上下文增强 → LLM生成。优势：实时知识更新、减少幻觉、提高可溯源性。
[rag_components] RAG 系统组件：数据源包括文档、数据库、API；分块处理包括文本分割、语义分块；向量化使用Embedding模型；向量数据库包括Milvus、Pinecone等；检索器做相似度搜索、重排序；生成器是LLM推理引擎。

用户问题：RAG 的工作流程和系统组件是什么？

请基于资料回答，缺失信息要说明。


## 7. 压缩与质量评估

上下文压缩的目标不是“把内容砍短”，而是“在预算更小的前提下保留更高的信息密度”。

最常见的错误是只保留开头几句，结果把真正影响回答的条件信息删掉了。例如制度文本中最重要的往往不是定义，而是：

- 时间
- 数值
- 否定条件
- 例外情况
- 来源

因此压缩之后还要评估。最简单的评估可以先检查：

- 关键术语是否还在
- 用户问题是否还被保留
- 与回答强相关的证据是否丢失

下面的代码只是一个最小检查器，但它已经能帮助学生形成“压缩后要验证”的习惯。


In [31]:
def compress_context(text, max_lines=5):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    important = []
    for line in lines:
        if line.endswith('：') or any(term in line for term in ['长期记忆', '检索资料', '用户问题', 'RAG', '检索']):
            important.append(line)
    return "\n".join(important[:max_lines])


def evaluate_context(prompt, required_terms):
    return {term: (term in prompt) for term in required_terms}

compressed = compress_context(prompt, max_lines=6)
print('压缩后的上下文:')
print(compressed)
print()
print('关键项保留情况:')
print(evaluate_context(compressed, ["RAG", "检索", "长期记忆", "用户问题"]))


压缩后的上下文:
长期记忆：
检索资料：
[rag_principle] RAG 检索增强生成：结合检索系统与语言模型，突破知识截止限制。工作流程：用户查询 → 向量检索 → 上下文增强 → LLM生成。优势：实时知识更新、减少幻觉、提高可溯源性。
[rag_components] RAG 系统组件：数据源包括文档、数据库、API；分块处理包括文本分割、语义分块；向量化使用Embedding模型；向量数据库包括Milvus、Pinecone等；检索器做相似度搜索、重排序；生成器是LLM推理引擎。
用户问题：RAG 的工作流程和系统组件是什么？

关键项保留情况:
{'RAG': True, '检索': True, '长期记忆': True, '用户问题': True}


## 7.1 最佳实践：成本监控与配置对比

这一节用课程 PPT 自身作为数据源，对应 slide13 的“最佳实践”和 slide12 的“上下文窗口管理”。

问题固定为：**如何讲解上下文工程最佳实践？**

这里不要只比较哪个配置 token 少。一个配置先要满足课程讲解需要，才值得比较成本：

- 最佳实践证据完整：是否覆盖“保持上下文简洁、监控token使用、测试不同上下文配置”。
- 格式化要求完整：是否覆盖“结构化格式、清晰分隔”。
- 是否有课程依据：是否至少包含一个真实 PPT slide 片段。

表格中的“合格”由这些条件自动计算。最后推荐的是“合格配置中 token 最少”的方案。


In [32]:
def estimate_tokens(text):
    chinese = sum(1 for c in text if "\u4e00" <= c <= "\u9fff")
    other = len(text) - chinese
    return int(chinese * 1.5 + other * 0.25)


question = '如何讲解上下文工程最佳实践？'
context_candidates = {
    'system_rule': {
        'type': 'system',
        'text': '你是上下文工程课程助手，只能依据课程PPT回答，并列出依据slide。',
    },
    'slide13_best_practices': {
        'type': 'ppt',
        'text': COURSE_SOURCES['slide13_best_practices'],
    },
    'slide12_token_budget': {
        'type': 'ppt',
        'text': COURSE_SOURCES['slide12_token_budget'],
    },
    'slide11_prompt_context': {
        'type': 'ppt',
        'text': COURSE_SOURCES['slide11_prompt_context'],
    },
    'slide14_scenarios': {
        'type': 'ppt',
        'text': COURSE_SOURCES['slide14_scenarios'],
    },
    'user_question': {
        'type': 'user',
        'text': f'用户问题：{question}',
    },
}

config_rules = {
    'minimal': ['system_rule', 'user_question'],
    'cost_only': ['system_rule', 'slide12_token_budget', 'user_question'],
    'scenario_only': ['system_rule', 'slide14_scenarios', 'user_question'],
    'balanced': ['system_rule', 'slide13_best_practices', 'slide12_token_budget', 'slide11_prompt_context', 'user_question'],
}

required_best_practices = ['保持上下文简洁', '监控token使用', '测试不同上下文配置']
required_format_terms = ['结构化格式', '清晰分隔']


def build_context(config_name):
    names = config_rules[config_name]
    items = [(name, context_candidates[name]) for name in names]
    text = '\n'.join(f"[{item['type']}:{name}] {item['text']}" for name, item in items)
    return items, text


def evaluate_config(config_name):
    items, context_text = build_context(config_name)
    practice_hits = sum(term in context_text for term in required_best_practices)
    format_hits = sum(term in context_text for term in required_format_terms)
    has_ppt_source = any(item['type'] == 'ppt' for _, item in items)
    qualified = practice_hits == len(required_best_practices) and format_hits == len(required_format_terms) and has_ppt_source
    tokens = estimate_tokens(context_text)
    cost = tokens / 1000 * 0.01
    missing = []
    if practice_hits != len(required_best_practices):
        missing.append('最佳实践证据不完整')
    if format_hits != len(required_format_terms):
        missing.append('格式化要求不完整')
    if not has_ppt_source:
        missing.append('没有课程PPT依据')
    return {
        '配置': config_name,
        'Token': tokens,
        '成本': cost,
        '最佳实践证据': f'{practice_hits}/{len(required_best_practices)}',
        '格式化要求': f'{format_hits}/{len(required_format_terms)}',
        '课程依据': has_ppt_source,
        '合格': qualified,
        '问题': '无' if qualified else '；'.join(missing),
        '上下文文本': context_text,
    }


In [33]:
results = [evaluate_config(name) for name in config_rules]

print('各配置实际注入的上下文：')
for result in results:
    print()
    print(f"[{result['配置']}]")
    print(result['上下文文本'])

print()
print('配置评估表：')
headers = ['配置', 'Token', '成本($)', '最佳实践证据', '格式化要求', '课程依据', '合格', '主要问题']
print('| ' + ' | '.join(headers) + ' |')
print('| ' + ' | '.join(['---'] * len(headers)) + ' |')
for result in results:
    row = [
        result['配置'],
        result['Token'],
        f"{result['成本']:.4f}",
        result['最佳实践证据'],
        result['格式化要求'],
        '是' if result['课程依据'] else '否',
        '是' if result['合格'] else '否',
        result['问题'],
    ]
    print('| ' + ' | '.join(map(str, row)) + ' |')

qualified = [result for result in results if result['合格']]
recommended = min(qualified, key=lambda x: x['Token'])

print()
print(f"推荐配置：{recommended['配置']}")
print('推荐理由：从表格看，它同时覆盖最佳实践证据 3/3、格式化要求 2/2，并且有课程PPT依据；在所有合格配置中 token 最少。')


各配置实际注入的上下文：

[minimal]
[system:system_rule] 你是上下文工程课程助手，只能依据课程PPT回答，并列出依据slide。
[user:user_question] 用户问题：如何讲解上下文工程最佳实践？

[cost_only]
[system:system_rule] 你是上下文工程课程助手，只能依据课程PPT回答，并列出依据slide。
[ppt:slide12_token_budget] 上下文窗口管理：Token预算规划策略包括预留空间、动态分配、溢出处理、成本监控。预留空间是为系统指令和检索结果预留固定token；动态分配是根据对话阶段调整用户/AI比例；溢出处理是在超过阈值时触发摘要或截断；成本监控是追踪token消耗，优化利用率。
[user:user_question] 用户问题：如何讲解上下文工程最佳实践？

[scenario_only]
[system:system_rule] 你是上下文工程课程助手，只能依据课程PPT回答，并列出依据slide。
[ppt:slide14_scenarios] 应用场景：智能客服需要多轮对话理解、个性化服务；代码助手需要项目上下文、代码解释生成；文档助手需要长文档摘要、问答系统；数据分析需要上下文理解、数据可视化建议。
[user:user_question] 用户问题：如何讲解上下文工程最佳实践？

[balanced]
[system:system_rule] 你是上下文工程课程助手，只能依据课程PPT回答，并列出依据slide。
[ppt:slide13_best_practices] 最佳实践：保持上下文简洁，避免冗余信息；使用结构化格式提高可读性；定期清理无用对话历史；监控token使用，控制成本；根据场景选择合适的上下文策略；测试不同上下文配置的效果。
[ppt:slide12_token_budget] 上下文窗口管理：Token预算规划策略包括预留空间、动态分配、溢出处理、成本监控。预留空间是为系统指令和检索结果预留固定token；动态分配是根据对话阶段调整用户/AI比例；溢出处理是在超过阈值时触发摘要或截断；成本监控是追踪token消耗，优化利用率。
[ppt:slide11_prompt_context] Prompt 工程与上下文：System P

## 8. 教学案例建议

这一节可以把抽象技术映射到具体教学场景。推荐课堂上不要只列名称，而是按“问题-上下文-策略”三元组来讲。

- 课程问答机器人：问题是教学安排、考试方式、政策规则；上下文是课程大纲、FAQ、通知；策略是 RAG + 引用 + 新鲜度检查。
- 论文阅读助手：问题是总结、对比、定位章节；上下文是论文章节、读书笔记、用户关注点；策略是分章节检索 + 摘要压缩。
- 编程助教：问题是报错定位、代码改写、规范解释；上下文是错误日志、代码片段、项目约束、历史尝试；策略是运行时上下文优先。
- 多轮项目助手：问题是继续之前任务；上下文是当前目标、历史决策、固定约束；策略是短期记忆 + 长期记忆联合检索。

如果课堂时间有限，可以统一用“软件学院课程助教机器人”这个主线把全 notebook 串起来。


## 9. 上下文类型分层：Input / Runtime / Compression / Isolation / Long-term

较新的上下文工程资料会把上下文分成多类：启动时注入的输入上下文、每次运行传入的运行时上下文、接近窗口上限时的压缩上下文、通过子任务隔离出的局部上下文，以及跨会话保留的长期记忆。

这个分类比“把所有资料拼进同一个 prompt”更适合长任务和 Agent 场景，因为复杂系统往往不是一次性思考，而是分步骤、分子任务推进。

要点是：不是所有信息都应该进入同一个上下文。很多时候应该先隔离，再汇总。


In [34]:
@dataclass
class ContextLayer:
    name: str
    scope: str
    examples: List[str]
    risk: str

layers = [
    ContextLayer("input_context", "每次启动固定加载", ["system prompt", "课程规则", "项目约定"], "过长会挤占用户任务空间"),
    ContextLayer("runtime_context", "单次运行动态传入", ["用户身份", "权限", "当前文件"], "权限上下文必须可信"),
    ContextLayer("compressed_context", "窗口接近上限时生成", ["对话摘要", "检索摘要"], "压缩可能丢失关键证据"),
    ContextLayer("isolated_context", "子任务/子代理内部", ["并行研究", "大文件分析"], "需要只返回结论和证据"),
    ContextLayer("long_term_memory", "跨会话持久化", ["用户偏好", "项目事实", "历史决策"], "写入频率和更新策略要评估"),
]

for layer in layers:
    print(f"{layer.name}: {layer.scope} | 示例={layer.examples} | 风险={layer.risk}")


input_context: 每次启动固定加载 | 示例=['system prompt', '课程规则', '项目约定'] | 风险=过长会挤占用户任务空间
runtime_context: 单次运行动态传入 | 示例=['用户身份', '权限', '当前文件'] | 风险=权限上下文必须可信
compressed_context: 窗口接近上限时生成 | 示例=['对话摘要', '检索摘要'] | 风险=压缩可能丢失关键证据
isolated_context: 子任务/子代理内部 | 示例=['并行研究', '大文件分析'] | 风险=需要只返回结论和证据
long_term_memory: 跨会话持久化 | 示例=['用户偏好', '项目事实', '历史决策'] | 风险=写入频率和更新策略要评估


## 10. RAG 检索评估：Hit Rate 与 MRR

只看最终回答是否像是正确的，往往不够。因为模型可能是凭已有常识答对的，而不是因为系统真的找到了正确证据。

因此在 RAG 里，应该先评估检索：

- `Hit Rate`：正确文档是否进入 Top-K。
- `MRR`：正确文档排得是否足够靠前。

这两个指标非常适合教学，因为它们能让学生明确区分：

- “没找到”是召回问题。
- “找到了但排后面”是排序问题。

当检索指标差时，直接继续调 prompt 往往没有意义。


In [35]:
eval_set = [
    {"query": "RAG 包含哪些流程？", "gold": "rag"},
    {"query": "长期记忆和短期记忆有什么区别？", "gold": "memory"},
    {"query": "如何减少上下文 token？", "gold": "compression"},
]


def ranked_retrieve(query, top_k=3):
    q = tokenize(query)
    scored = []
    for name, text in docs.items():
        score = len(q & tokenize(text)) / max(len(q), 1)
        scored.append((score, name, text))
    return sorted(scored, key=lambda x: x[0], reverse=True)[:top_k]


def evaluate_retriever(dataset, top_k=2):
    hits = []
    reciprocal_ranks = []
    for item in dataset:
        ranked = ranked_retrieve(item["query"], top_k=top_k)
        names = [name for _, name, _ in ranked]
        hit = item["gold"] in names
        hits.append(hit)
        if hit:
            reciprocal_ranks.append(1 / (names.index(item["gold"]) + 1))
        else:
            reciprocal_ranks.append(0)
        print(item["query"], "=>", names, "gold=", item["gold"], "hit=", hit)
    return {"hit_rate": sum(hits) / len(hits), "mrr": sum(reciprocal_ranks) / len(reciprocal_ranks)}

print(evaluate_retriever(eval_set, top_k=2))


RAG 包含哪些流程？ => ['rag_principle', 'rag_components'] gold= rag hit= False
长期记忆和短期记忆有什么区别？ => ['rag_principle', 'rag_components'] gold= memory hit= False
如何减少上下文 token？ => ['context_injection', 'token_budget'] gold= compression hit= False
{'hit_rate': 0.0, 'mrr': 0.0}


## 11. 引用、新鲜度与来源覆盖检查

生产 RAG 不应只返回答案，还要返回来源和更新时间。这里不使用虚构政策，而是检查课程 PPT 中两个真实来源：slide12 和 slide13。

这个例子要说明三点：

- 引用：回答应能指出依据来自哪个 slide。
- 新鲜度：同一文件中的不同 slide 也应保留文件更新时间，便于追踪版本。
- 覆盖度：不同来源覆盖的知识点不同，应检查它们是否覆盖当前问题需要的概念。


In [36]:
from dataclasses import dataclass
from pathlib import Path
from datetime import datetime


@dataclass
class SourceChunk:
    source: str
    content: str
    updated_at: str


def find_course_file(relative_path):
    """Find a course file whether the notebook runs from repo root or examples/."""
    current = Path.cwd().resolve()
    candidates = [current / relative_path]
    candidates += [parent / relative_path for parent in current.parents]
    candidates += [current / 'context-engineering-course' / relative_path]
    candidates += [parent / 'context-engineering-course' / relative_path for parent in current.parents]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f'Cannot find {relative_path} from {current}')


def file_mtime(path):
    ts = Path(path).stat().st_mtime
    return datetime.fromtimestamp(ts).strftime('%Y-%m-%d %H:%M')

ppt_path = find_course_file('docs/上下文工程.pptx')

source_chunks = [
    SourceChunk(
        'docs/上下文工程.pptx#slide12',
        COURSE_SOURCES['slide12_token_budget'],
        file_mtime(ppt_path),
    ),
    SourceChunk(
        'docs/上下文工程.pptx#slide13',
        COURSE_SOURCES['slide13_best_practices'],
        file_mtime(ppt_path),
    ),
]

coverage_groups = {
    '成本监控': ['成本监控', '监控token使用', '追踪token消耗'],
    '结构化格式': ['结构化格式', '清晰分隔'],
    '配置测试': ['测试不同上下文配置'],
    '预算策略': ['预留空间', '动态分配', '溢出处理'],
}


def build_cited_context(chunks):
    lines = []
    for idx, chunk in enumerate(chunks, 1):
        lines.append(f'[{idx}] source={chunk.source} updated={chunk.updated_at}')
        lines.append(chunk.content)
    return '\n'.join(lines)


def coverage_check(chunks, groups):
    rows = []
    for chunk in chunks:
        row = {'source': chunk.source}
        for name, terms in groups.items():
            row[name] = any(term in chunk.content for term in terms)
        rows.append(row)
    return rows

print(build_cited_context(source_chunks))
print()
print('来源覆盖检查：')
headers = ['source'] + list(coverage_groups)
print('| ' + ' | '.join(headers) + ' |')
print('| ' + ' | '.join(['---'] * len(headers)) + ' |')
for row in coverage_check(source_chunks, coverage_groups):
    values = [row['source']] + ['是' if row[name] else '否' for name in coverage_groups]
    print('| ' + ' | '.join(values) + ' |')
print()
print('建议：回答最佳实践问题时优先引用 slide13；讲 token 预算和成本监控细节时补充 slide12。')


[1] source=docs/上下文工程.pptx#slide12 updated=2026-05-11 16:36
上下文窗口管理：Token预算规划策略包括预留空间、动态分配、溢出处理、成本监控。预留空间是为系统指令和检索结果预留固定token；动态分配是根据对话阶段调整用户/AI比例；溢出处理是在超过阈值时触发摘要或截断；成本监控是追踪token消耗，优化利用率。
[2] source=docs/上下文工程.pptx#slide13 updated=2026-05-11 16:36
最佳实践：保持上下文简洁，避免冗余信息；使用结构化格式提高可读性；定期清理无用对话历史；监控token使用，控制成本；根据场景选择合适的上下文策略；测试不同上下文配置的效果。

来源覆盖检查：
| source | 成本监控 | 结构化格式 | 配置测试 | 预算策略 |
| --- | --- | --- | --- | --- |
| docs/上下文工程.pptx#slide12 | 是 | 否 | 否 | 是 |
| docs/上下文工程.pptx#slide13 | 是 | 是 | 是 | 否 |

建议：回答最佳实践问题时优先引用 slide13；讲 token 预算和成本监控细节时补充 slide12。
